# 03 — Iterative Debugging Loop & Execution Reward Design
**Goal**: Implement the multi-turn agentic debugging loop ($K=3$ turns) with error feedback (Verification Check V2), define execution-guided reward functions, synthetic bug injection, and collect trajectory preference pairs for DPO.

---

## Step 1: Environment & Module Setup

In [ ]:
import sys, os, shutil

# 1. Clear any previously cached 'src' modules from Python memory
for mod_name in list(sys.modules.keys()):
    if mod_name == "src" or mod_name.startswith("src."):
        del sys.modules[mod_name]

# 2. Prepare working copy of 'src' in /kaggle/working
input_src = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "src" in dirs:
        input_src = os.path.join(root, "src")
        break

working_src = "/kaggle/working/src"
if input_src:
    if os.path.exists(working_src):
        shutil.rmtree(working_src)
    shutil.copytree(input_src, working_src)
    print(f"SUCCESS: Prepared working copy of 'src' at {working_src}")

# 3. Set up import paths without overwriting repository source files
if os.path.exists(working_src) and "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")
elif os.path.abspath(".") not in sys.path:
    sys.path.insert(0, os.path.abspath("."))

# 4. Import all modules
import torch
from datasets import load_dataset
from src.models.loader import load_model_and_tokenizer
from src.debugging.debug_loop import build_prompt, agentic_debug_loop, DebugLoop
from src.rewards.execution_reward import compute_reward, compute_reward_binary, compute_partial_reward, compute_status_aware_reward
from src.training.dpo import make_preference_pairs
from src.error_injection import BugInjector, BugType

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment & modules initialized successfully!")

## Step 2: Synthetic Bug Injection Demo

In [ ]:
injector = BugInjector(seed=42)
correct_code = "def multiply(a, b):\n    return a * b"

print("=== Synthetic Bug Injection ===")
for category in list(BugType):
    res = injector.inject_bug(correct_code, category=category)
    print(f"[{res['bug_type'].upper()}] -> {res['description']}")

## Step 3A: Controlled Bug-Injection Self-Correction (Verification Check V2 — Part A)
Injects a synthetic logic bug into Turn 1 code to produce a guaranteed `WA`/`RE` failure $\rightarrow$ captures traceback $\rightarrow$ model receives execution feedback in Turn 2 and generates the fix to reach `AC`.

In [ ]:
!pip uninstall -y torchao

import sys, os
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"
print(f"Loading model and tokenizer: {MODEL_NAME} in FP16 precision...")
model, tokenizer = load_model_and_tokenizer(
    model_name=MODEL_NAME,
    load_in_4bit=False,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    attach_lora=True,
)

humaneval = load_dataset('openai_humaneval', split='test')
prob0 = humaneval[0]
problem_stmt = prob0['prompt']
test_cases0 = [prob0['test'] + f"\ncheck({prob0['entry_point']})\n"]

# 1. Create a controlled buggy solution for Turn 1
correct_ref = """from typing import List

def has_close_elements(numbers: List[float], threshold: float) -> bool:
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if abs(numbers[i] - numbers[j]) < threshold:
                return True
    return False
"""

injector = BugInjector(seed=42)
bug_res = injector.inject_bug(correct_ref, category=BugType.LOGIC)
buggy_turn1_code = bug_res['buggy_code']

print("=== Verification Check V2 (Part A: Controlled Self-Correction Trajectory) ===")
print(f"Injected Bug Description: {bug_res['description']}")

trajectory_a = agentic_debug_loop(model, tokenizer, problem_stmt, test_cases=test_cases0, initial_code=buggy_turn1_code, K=2)

for turn_info in trajectory_a:
    print(f"\n--- [Turn {turn_info['turn']}] ---")
    print(f"Status: {turn_info['result']['status']}")
    print(f"Code Snippet:\n{turn_info['code'][:180]}...")
    if turn_info['result'].get('traceback'):
        print(f"Feedback / Traceback:\n{turn_info['result']['traceback'][:200]}...")

print("\nVerification Check V2 Part A completed!")

## Step 3B: Natural Zero-Shot Failure Self-Correction (Verification Check V2 — Part B)
Evaluates a problem where model zero-shot generation naturally fails on Turn 1 $\rightarrow$ receives test failure feedback $\rightarrow$ self-corrects on Turn 2 to reach `AC`.

In [ ]:
# Problem 10 in HumanEval requires specific string palindrome logic that can fail zero-shot
prob10 = humaneval[10]
problem_b_stmt = prob10['prompt']
test_cases10 = [prob10['test'] + f"\ncheck({prob10['entry_point']})\n"]

print("=== Verification Check V2 (Part B: Natural Failure Trajectory) ===")
print(f"Problem: {prob10['entry_point']}")

trajectory_b = agentic_debug_loop(model, tokenizer, problem_b_stmt, test_cases=test_cases10, K=3)

for turn_info in trajectory_b:
    print(f"\n--- [Turn {turn_info['turn']}] ---")
    print(f"Status: {turn_info['result']['status']}")
    print(f"Code Snippet:\n{turn_info['code'][:180]}...")
    if turn_info['result'].get('traceback'):
        print(f"Feedback / Traceback:\n{turn_info['result']['traceback'][:200]}...")

print("\nVerification Check V2 Part B completed!")

## Step 4: Execution Reward Function Verification
Validates reward scores for AC ($+1.0$), WA/partial pass ($0.0 - 1.0$), and CE/RE ($-0.2$ penalty).

In [ ]:
print("--- Dense Reward Calculations ---")
print(f"AC (5/5 tests): {compute_reward('AC', 5, 5)}")
print(f"WA (3/5 tests): {compute_reward('WA', 3, 5)}")
print(f"CE (0/5 tests): {compute_reward('CE', 0, 5)}")
print(f"RE (0/5 tests): {compute_reward('RE', 0, 5)}")

print("\n--- Binary Reward Calculations (RQ4 Ablation) ---")
print(f"AC (5/5 tests): {compute_reward_binary('AC', 5, 5)}")
print(f"WA (3/5 tests): {compute_reward_binary('WA', 3, 5)}")

## Step 5: Preference Pair Generation for DPO
Collects `(prompt, chosen, rejected)` trajectory pairs from debug rollouts for DPO training.

In [ ]:
print("Generating sample DPO preference pairs...")

eval_data = load_dataset('openai_humaneval', split='test[:5]')
pref_data = make_preference_pairs(eval_data, model, tokenizer, K=3)
print(f"Total preference pairs collected: {len(pref_data)}")
if len(pref_data) > 0:
    print(f"Sample Chosen:\n{pref_data[0]['chosen'][:100]}...")
    print(f"Sample Rejected:\n{pref_data[0]['rejected'][:100]}...")